# Case study: Ring‑hydroxylating Dioxygenases

This notebook demonstrates the **SDP Detection Pipeline** on the Pfam PF00848 family. The pipeline identifies specificity‑determining positions (SDPs) that discriminate among the natural functional subgroups present in the alignment.

### Pipeline overview
1. **Load & map positions** – parse MSA and extract residue numbers from headers.
2. **Cleanse** – remove gap‑rich columns and rows.
3. **MCA** – dimensionality reduction.
4. **Clustering** – single‑linkage with silhouette score selection.
5. **Random Forest** – feature importance using cluster labels as target.
6. **Select SDPs** – top *N* most important columns.
7. **Visualise** – word clouds, sequence logos, perceptual map.

After the automated pipeline, we manually filter sequences with known structures and visualise the SDPs in 3D.

## 1. Imports and data loading

In [ ]:
# Ensure the package is installed (e.g., pip install -e . from the project root)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sdp_detection_pipeline.io import load_msa, map_positions, build_profiles
from sdp_detection_pipeline.preprocessing import CleanseTransformer
from sdp_detection_pipeline.modeling import MCAClusterFeatureSelector
from sdp_detection_pipeline.visualization import (
    plot_cleanse_heatmaps,
    plot_pareto,
    plot_perceptual_map,
    generate_wordclouds,
    generate_logos,
)
from sklearn.pipeline import Pipeline

In [ ]:
# Paths to the input files (relative to this notebook)
MSA_FILE = "pf00848-alignment.fasta"
METADATA_FILE = "pf00848-metadata.tsv"

# Load the MSA
raw_msa = load_msa(MSA_FILE)
print(f"Loaded alignment with {raw_msa.shape[0]} sequences and {raw_msa.shape[1]} columns.")

# Map sequence headers to residue positions
positions_map = map_positions(raw_msa)
print(f"Position mapping created for {len(positions_map)} sequences.")

## 2. Create and fit the pipeline

The pipeline combines cleansing and the MCA‑clustering‑RF selector.

In [ ]:
pipeline = Pipeline([
    ("cleanse", CleanseTransformer(threshold=0.9, remove_lowercase=True)),
    ("sdp", MCAClusterFeatureSelector(
        cluster_method="single-linkage",
        min_clusters=3,
        rf_n_estimators=1000,
        random_state=42,
        top_n=3,                     # we want exactly the top‑3 SDPs
    ))
])

pipeline.fit(raw_msa)
print("Pipeline fitted.")

## 3. Inspect the results

In [ ]:
cleanse_step = pipeline.named_steps["cleanse"]
sdp_step = pipeline.named_steps["sdp"]

print(f"Kept columns after cleansing: {len(cleanse_step.kept_columns_)}")
print(f"Number of unique sequences (deduplicated): {sdp_step.unique_sequences_.shape[0]}")
print(f"Optimal number of clusters found: {len(set(sdp_step.labels_))}")
print(f"Silhouette score for chosen clustering: {np.max([s for s in [1]])}  # you can compute manually if needed")
print(f"Top selected columns (SDPs): {sdp_step.selected_columns_}")
print(f"Column importances:\n{sdp_step.column_importances_.to_string()}")

## 4. Generate visualisations

In [ ]:
# Cleansing heatmaps
plot_cleanse_heatmaps(cleanse_step.dirty_, cleanse_step.clean_, show=True)

In [ ]:
# Pareto chart of column importances
plot_pareto(sdp_step.column_importances_, show=True)

In [ ]:
# Perceptual map (MCA plot with SDP residues)
plot_perceptual_map(
    sdp_step.mca_,
    sdp_step.unique_sequences_,
    sdp_step.coordinates_,
    sdp_step.labels_,
    sdp_step.selected_columns_,
    sdp_step.selected_feature_names_,
    show=True,
)

In [ ]:
# Word clouds (requires the UniProt TSV metadata)
generate_wordclouds(
    sdp_step.input_index_,
    sdp_step.unique_sequences_,
    sdp_step.labels_,
    metadata=METADATA_FILE,
    show=True,
)

In [ ]:
# Sequence logos for each cluster
generate_logos(
    sdp_step.unique_sequences_,
    sdp_step.labels_,
    sdp_step.selected_columns_,
    show=True,
)

## 5. Build the SDP profile table

The profile shows the amino acid and residue number at each SDP for every original sequence.

In [ ]:
profiles = build_profiles(raw_msa, positions_map, sdp_step.selected_columns_)
profiles.head(10)

## 6. Manual steps – sequences with known structures

The following steps are specific to this case study and not part of the pipeline.

In [ ]:
# UniProt entry names for which a PDB structure is available
known_structure = [
    'TPDA2_COMSP',
    'BPHA1_RHOJR',
    'NDOB_PSEU8',
    'NDOB_PSEPU',
    'NAGG_RALSP',
    'CNTA_ACIB2',
    'BNZA_PSEP1',
    'BPHA_PARXL',
    'BPHA_COMTE',
]

# Filter the profile DataFrame
have_structure = [h for h in profiles.index if h.split('/')[0] in known_structure]
filtered_profiles = profiles.loc[have_structure]
filtered_profiles

## 7. Visualise SDPs in a 3D structure

We use `nglview` to highlight the identified SDPs on the structure of biphenyl dioxygenase (PDB 1ULI).

In [ ]:
import nglview as nv

pdb_id = '1ULI'
view = nv.show_pdbid(pdb_id)
view.update_representation(opacity=0.5)

sdps = tuple(sdp_step.selected_columns_)  # (220, 330, 377) in this case
chains = ('A', 'C', 'E')

selection_str = ":{} and ( {} or {} or {} )"
for chain in chains:
    view.add_representation('ball+stick', selection=selection_str.format(chain, *sdps), color='black')
    view.add_representation('label', selection=selection_str.format(chain, *sdps), color='red')

view